<a href="https://colab.research.google.com/github/Soorajr005/Lightweight-Vision-Transformers-for-Edge-AI-and-Real-Time-Object-Detection/blob/main/Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install ultralytics onnx onnxruntime-gpu tabulate -q

import torch
if not torch.cuda.is_available():
    raise SystemError("GPU not found! Go to Runtime > Change runtime type > select T4 GPU.")

print("=" * 45)
print(f"CUDA Available : {torch.cuda.is_available()}")
print(f"Device Name    : {torch.cuda.get_device_name(0)}")
print("=" * 45)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.7/246.7 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobu

In [ ]:
from ultralytics import YOLO

# 1. Loading pretrained YOLOv8n (Nano - lightweight CNN)
cnn_model = YOLO("yolov8n.pt")

# 2. Train on Pascal VOC dataset
# imgsz=416: optimized resolution for low-latency edge deployment
cnn_results = cnn_model.train(
    data="VOC.yaml",
    epochs=10,
    imgsz=416,
    batch=32,
    device=0,
    name="cnn_yolov8n_voc"
)

# 3. Evaluate model on validation set
metrics = cnn_model.val()

p = metrics.box.mp
r = metrics.box.mr
f1 = 2 * (p * r) / (p + r + 1e-8)
map50 = metrics.box.map50
map50_95 = metrics.box.map

print("\n" + "=" * 50)
print("     CNN BASELINE (YOLOv8n) RESULTS")
print("=" * 50)
print(f"Precision   : {p:.4f}")
print(f"Recall      : {r:.4f}")
print(f"F1-Score    : {f1:.4f}")
print(f"mAP@50      : {map50:.4f}")
print(f"mAP@50-95   : {map50_95:.4f}")
print("=" * 50)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VOC.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0

In [ ]:
from google.colab import files
import os
onnx_path = cnn_model.export(format="onnx", imgsz=416, dynamic=False)
file_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)

print("\n" + "=" * 50)
print(f"Exported ONNX Path : {onnx_path}")
print(f"ONNX Model Size    : {file_size_mb:.2f} MB")
print("=" * 50)

# 2. Downloadng the ONNX file
files.download(onnx_path)

Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 72 layers, 3,009,548 parameters, 0 gradients, 3.4 GFLOPs

PyTorch: starting from '/content/runs/detect/cnn_yolov8n_voc/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 24, 3549) (5.9 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 10 packages in 287ms
Prepared 2 packages in 33ms
Installed 2 packages in 8ms
 + colorama==0.4.6
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 2.3s, saved as '/content/run

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from ultralytics import RTDETR

# 1. Load Pretrained RT-DETR Vision Transformer
vit_model = RTDETR("rtdetr-l.pt")

# 2. Train on Pascal VOC
# batch=16 fits the larger Transformer attention matrices into memory
vit_results = vit_model.train(
    data="VOC.yaml",
    epochs=5,
    imgsz=416,
    batch=16,
    device=0,
    name="vit_rtdetr_voc"
)

# 3. Evaluate Transformer metrics
vit_metrics = vit_model.val()

p_v = vit_metrics.box.mp
r_v = vit_metrics.box.mr
f1_v = 2 * (p_v * r_v) / (p_v + r_v + 1e-8)
map50_v = vit_metrics.box.map50
map50_95_v = vit_metrics.box.map

print("\n" + "=" * 50)
print("     TRANSFORMER (RT-DETR) RESULTS")
print("=" * 50)
print(f"Precision   : {p_v:.4f}")
print(f"Recall      : {r_v:.4f}")
print(f"F1-Score    : {f1_v:.4f}")
print(f"mAP@50      : {map50_v:.4f}")
print(f"mAP@50-95   : {map50_95_v:.4f}")
print("=" * 50)

# 4. Export Transformer to ONNX
vit_onnx_path = vit_model.export(format="onnx", imgsz=416, dynamic=False)
files.download(vit_onnx_path)

Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VOC.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=vit_rtdetr_voc, nbs=64, nms=None, opset=None, optimize=False, optim

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        1/5      7.96G     0.3082      1.408     0.2766         13        416: 100% ━━━━━━━━━━━━ 1035/1035 1.7it/s 10:08
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 155/155 2.8it/s 55.5s
                   all       4952      12032      0.811      0.807      0.858      0.677

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        2/5      7.36G     0.2036     0.5386     0.2503         33        416: 0% ──────────── 0/1035  0.6s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


        2/5      7.36G     0.2846     0.5437     0.2549         17        416: 100% ━━━━━━━━━━━━ 1035/1035 1.7it/s 9:58
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 155/155 2.9it/s 54.4s
                   all       4952      12032      0.817      0.788      0.857      0.675

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        3/5       7.3G     0.3537     0.5518     0.1891         38        416: 0% ──────────── 0/1035  0.6s

...

        5/5      8.96G     0.2526     0.4383     0.2193         18        416: 100% ━━━━━━━━━━━━ 1035/1035 1.7it/s 9:58
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 155/155 2.8it/s 54.7s
                   all       4952      12032      0.838      0.839      0.888      0.713

5 epochs completed in 0.920 hours.
Optimizer stripped from /content/runs/detect/vit_rtdetr_voc/weights/last.pt, 66.3MB
Optimizer stripped from /content/runs/detect/vit_rtdetr_voc/weights/best.pt, 66.3MB

Validating /content/runs/detect/vit_rtdetr_voc/weights/best.pt...
Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary (fused): 315 layers, 32,024,840 parameters, 0 gradients, 105.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 155/155 2.9it/s 54.3s
                   all       4952      12032      0.839      0.838      0.888      0.712

/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not applied. (Triggered internally at /pytorch/torch/csrc/jit/passes/onnx/constant_fold.cpp:178.)
  _C._jit_pass_onnx_node_shape_type_inference(node, params_dict, opset_version)
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 18 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:714: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice

ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 16.8s, saved as '/content/runs/detect/vit_rtdetr_voc/weights/best.onnx' (125.1 MB)

Export complete (19.1s)
Results saved to /content/runs/detect/vit_rtdetr_voc/weights/best.onnx
Predict:         yolo predict task=detect model=/content/runs/detect/vit_rtdetr_voc/weights/best.onnx imgsz=416 
Validate:        yolo val task=detect model=/content/runs/detect/vit_rtdetr_voc/weights/best.onnx imgsz=416 data=/usr/local/lib/python3.13/dist-packages/ultralytics/cfg/datasets/VOC.yaml  
Visualize:       https://netron.app


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>